# End-to-End Probe Inference Demo

This notebook walks through the Evee pathogenicity prediction pipeline:
1. Load pre-extracted Evo2 activations for 8 ClinVar variants
2. Inspect the activation tensor layout
3. Run the covariance probe with Newton-Schulz spectral compression
4. Compare predictions to ground truth

**No GPU required.** All artifacts are pre-computed and included in the repo.
Only standard libraries: `torch`, `safetensors`, `polars`.

In [1]:
from pathlib import Path

import polars as pl
import safetensors.torch
import torch
from torch import Tensor, nn

SAMPLES = Path("../artifacts/samples")

## 1. Load Sample Variants

8 curated ClinVar variants from the gene-holdout test set:
4 pathogenic + 4 benign, across missense, splice, intronic, and synonymous.

In [2]:
variants = pl.read_ipc(SAMPLES / "variants.feather")
variants

variant_id,gene_name,chrom,pos,ref,alt,consequence,label
str,str,str,i64,str,str,str,str
"""chr17:50196171:C:G""","""COL1A1""","""chr17""",50196171,"""C""","""G""","""missense_variant""","""pathogenic"""
"""chr17:83079863:C:G""","""METRNL""","""chr17""",83079863,"""C""","""G""","""missense_variant""","""benign"""
"""chr16:30188087:G:A""","""CORO1A""","""chr16""",30188087,"""G""","""A""","""splice_donor_variant""","""pathogenic"""
"""chr1:159953443:C:T""","""SLAMF9""","""chr1""",159953443,"""C""","""T""","""splice_donor_variant""","""benign"""
"""chrX:20169502:A:C""","""RPS6KA3""","""chrX""",20169502,"""A""","""C""","""intron_variant""","""pathogenic"""
"""chr15:78162452:T:C""","""IDH3A""","""chr15""",78162452,"""T""","""C""","""intron_variant""","""benign"""
"""chr10:71799628:G:A""","""CDH23""","""chr10""",71799628,"""G""","""A""","""synonymous_variant""","""pathogenic"""
"""chr4:122947300:C:T""","""AFG2A""","""chr4""",122947300,"""C""","""T""","""synonymous_variant""","""benign"""


## 2. Load Activations

Each variant has its own safetensors file containing a `[2, 2, 256, 4096]` tensor:

| Dim | Size | Meaning |
|-----|------|---------|
| 0   | 2    | Direction: forward, backward (reverse complement) |
| 1   | 2    | View: variant sequence, reference sequence |
| 2   | 256  | Positions selected by cosine divergence |
| 3   | 4096 | Evo2-7B block 27 hidden dimension |

The 256 positions are where the model's internal representation differs most
between variant and reference — where it "notices" the mutation.

In [3]:
variant_ids = variants["variant_id"].to_list()
activations = torch.stack([
    safetensors.torch.load_file(str(SAMPLES / f"{vid.replace(':', '_')}.safetensors"))["activations"]
    for vid in variant_ids
]).float()

print(f"Loaded {activations.shape[0]} variants: {activations.shape}")

Loaded 8 variants: torch.Size([8, 2, 2, 256, 4096])


In [4]:
# Inspect the first variant: COL1A1 missense pathogenic
var_fwd = activations[0, 0, 0]   # variant, forward direction
ref_fwd = activations[0, 0, 1]   # reference, forward direction
diff = var_fwd - ref_fwd

print(f"Variant: {variant_ids[0]} ({variants[0, 'gene_name']}, {variants[0, 'consequence']})")
print(f"  Variant activation norm:  {var_fwd.norm(dim=-1).mean():.2f}")
print(f"  Reference activation norm: {ref_fwd.norm(dim=-1).mean():.2f}")
print(f"  Difference norm:           {diff.norm(dim=-1).mean():.4f}")

Variant: chr17:50196171:C:G (COL1A1, missense_variant)
  Variant activation norm:  29.84
  Reference activation norm: 29.66
  Difference norm:           8.8840


## 3. Covariance Probe

The probe architecture has three stages:

1. **Untied projection**: Two separate linear maps (left, right) per direction
   project each position from 4096-d to 64-d
2. **Covariance pooling**: `left.T @ right / K` captures second-order
   interactions across all 256 positions → a 64×64 matrix per direction.
   Diagonal regularization (`εI`) ensures stability, then **Newton-Schulz
   spectral compression** (matrix square root) dampens dominant eigenvalues.
3. **Factored readout**: A factored contraction on the compressed
   covariance produces pathogenicity logits.

```
diff_d = var_d - ref_d                         per direction d ∈ {fwd, bwd}
left_d, right_d = proj_left_d(diff_d),         [B, K, 64]
                  proj_right_d(diff_d)
cov = Σ_d left_d.T @ right_d / K + εI          [B, 64, 64]
cov = sqrtm(cov)                                Newton-Schulz (3 iterations)
feat = W_left @ cov @ W_right.T                 [B, 128]
logits = out(feat)                              [B, 2]
```

In [5]:
def newton_schulz_sqrtm(matrix: Tensor, n_iters: int = 3) -> Tensor:
    """Matrix square root via Newton-Schulz coupled iteration.

    Normalizes by Frobenius norm to bound eigenvalues to (0, 1],
    runs the iteration, then rescales.
    """
    dtype = matrix.dtype
    m = matrix.float()
    d = m.size(-1)

    frob = m.flatten(-2).norm(dim=-1)[..., None, None]
    scale = frob.sqrt()
    normed = m / frob

    eye = torch.eye(d, device=m.device)
    y, z = normed, eye.expand_as(m).clone()
    for _ in range(n_iters):
        t = 3 * eye - z @ y
        y = y @ t / 2
        z = t @ z / 2

    return (y * scale).to(dtype)


class CovarianceProbe(nn.Module):
    """Covariance probe for bidirectional genomic activations.

    Pools variable-length activation diffs into a cross-covariance matrix
    via untied left/right projections, applies Newton-Schulz spectral
    compression, then classifies with a factored readout.
    """

    def __init__(self, d_model: int = 4096, d_hidden: int = 64,
                 d_probe: int = 128, n_outputs: int = 2,
                 n_sqrtm_iters: int = 3, eps: float = 1e-3):
        super().__init__()
        self.n_sqrtm_iters = n_sqrtm_iters
        self.eps = eps

        self.proj_left_fwd = nn.Linear(d_model, d_hidden)
        self.proj_right_fwd = nn.Linear(d_model, d_hidden)
        self.proj_left_bwd = nn.Linear(d_model, d_hidden)
        self.proj_right_bwd = nn.Linear(d_model, d_hidden)
        self.register_buffer("_eye", torch.eye(d_hidden))

        self.head_left = nn.Linear(d_hidden, d_probe, bias=False)
        self.head_right = nn.Linear(d_hidden, d_probe, bias=False)
        self.out = nn.Linear(d_probe, n_outputs)

    def embedding(self, activations: Tensor) -> Tensor:
        """[B, 2, 2, K, d] → [B, d_hidden, d_hidden] covariance matrix."""
        diff_fwd = activations[:, 0, 0] - activations[:, 0, 1]
        diff_bwd = activations[:, 1, 0] - activations[:, 1, 1]

        left_fwd = self.proj_left_fwd(diff_fwd).float()
        right_fwd = self.proj_right_fwd(diff_fwd).float()
        left_bwd = self.proj_left_bwd(diff_bwd).float()
        right_bwd = self.proj_right_bwd(diff_bwd).float()

        K = left_fwd.shape[1]
        cov = (
            torch.bmm(left_fwd.transpose(1, 2), right_fwd) / K
            + torch.bmm(left_bwd.transpose(1, 2), right_bwd) / K
            + self.eps * self._eye
        )
        return newton_schulz_sqrtm(cov, self.n_sqrtm_iters)

    def forward(self, activations: Tensor) -> Tensor:
        """[B, 2, 2, K, d] → [B, n_outputs] logits."""
        cov = self.embedding(activations)
        feat = torch.einsum(
            "blr,hl,hr->bh", cov,
            self.head_left.weight, self.head_right.weight,
        )
        return self.out(feat)

    def predict(self, activations: Tensor) -> Tensor:
        """[B, 2, 2, K, d] → [B] P(pathogenic)."""
        return torch.softmax(self(activations), dim=-1)[:, 1]

In [6]:
probe = CovarianceProbe()
safetensors.torch.load_model(probe, str(SAMPLES / "probe.safetensors"))
probe.eval()
print(probe)

CovarianceProbe(
  (proj_left_fwd): Linear(in_features=4096, out_features=64, bias=True)
  (proj_right_fwd): Linear(in_features=4096, out_features=64, bias=True)
  (proj_left_bwd): Linear(in_features=4096, out_features=64, bias=True)
  (proj_right_bwd): Linear(in_features=4096, out_features=64, bias=True)
  (head_left): Linear(in_features=64, out_features=128, bias=False)
  (head_right): Linear(in_features=64, out_features=128, bias=False)
  (out): Linear(in_features=128, out_features=2, bias=True)
)


## 4. Run Inference

In [7]:
with torch.no_grad():
    scores = probe.predict(activations)

results = variants.with_columns(
    pl.Series("score", [round(s, 4) for s in scores.tolist()]),
    pl.Series("prediction", ["pathogenic" if s > 0.5 else "benign" for s in scores.tolist()]),
).with_columns(
    (pl.col("prediction") == pl.col("label")).alias("correct"),
)
results.select("variant_id", "gene_name", "consequence", "label", "score", "prediction", "correct")

variant_id,gene_name,consequence,label,score,prediction,correct
str,str,str,str,f64,str,bool
"""chr17:50196171:C:G""","""COL1A1""","""missense_variant""","""pathogenic""",0.9999,"""pathogenic""",true
"""chr17:83079863:C:G""","""METRNL""","""missense_variant""","""benign""",0.0159,"""benign""",true
"""chr16:30188087:G:A""","""CORO1A""","""splice_donor_variant""","""pathogenic""",0.9771,"""pathogenic""",true
"""chr1:159953443:C:T""","""SLAMF9""","""splice_donor_variant""","""benign""",0.025,"""benign""",true
"""chrX:20169502:A:C""","""RPS6KA3""","""intron_variant""","""pathogenic""",0.823,"""pathogenic""",true
"""chr15:78162452:T:C""","""IDH3A""","""intron_variant""","""benign""",0.0822,"""benign""",true
"""chr10:71799628:G:A""","""CDH23""","""synonymous_variant""","""pathogenic""",0.8829,"""pathogenic""",true
"""chr4:122947300:C:T""","""AFG2A""","""synonymous_variant""","""benign""",0.0021,"""benign""",true


In [8]:
n_correct = results["correct"].sum()
print(f"Accuracy: {n_correct}/{results.height} ({100 * n_correct / results.height:.0f}%)")

Accuracy: 8/8 (100%)


## Summary

The pipeline extracts Evo2-7B block 27 activations at 256 divergent
positions per variant, computes the variant−reference diff from both
reading directions, pools via cross-covariance with Newton-Schulz
spectral compression, and classifies with a factored readout.

The full model achieves **0.97 AUROC** on 34K held-out variants
(gene-level split), outperforming CADD, AlphaMissense, and other
computational predictors across all consequence types.